# Bronze DQ — Great Expectations Validation
**CNG Distribution Analytics Platform**

Validates `Files/bronzedata/` output before Silver is allowed to run.

## Checks
| Category | Check |
|---|---|
| Completeness | Row count > 0 |
| Completeness | Row count regression vs baseline |
| Schema | All expected columns present |
| Schema | No unexpected extra columns (WARN only) |
| Schema | Column count matches expected |
| Hygiene | Critical ID columns non-null |
| Hygiene | Audit columns present and fully populated |

## Pipeline position
```
ForEach → Bronze notebook → THIS notebook → Silver notebook
```

## 1. Config

In [9]:
import os, json
import pandas as pd
from datetime import datetime, timezone

BRONZE_PATH   = "abfss://CNG_project@onelake.dfs.fabric.microsoft.com/bronze_lakehouse.Lakehouse/Files/bronzedata/"
BASELINE_PATH = "abfss://CNG_project@onelake.dfs.fabric.microsoft.com/bronze_lakehouse.Lakehouse/Files/dq_baseline/"
DQ_LOG_PATH   = "abfss://CNG_project@onelake.dfs.fabric.microsoft.com/bronze_lakehouse.Lakehouse/Files/dq_logs/"

for p in [BASELINE_PATH, DQ_LOG_PATH]:
    os.makedirs(p, exist_ok=True)

run_at          = datetime.now(timezone.utc)
pipeline_run_id = run_at.strftime('%Y%m%d_%H%M%S')

# Audit columns stamped by the Bronze ingestion notebook
AUDIT_COLS = ["_ingested_at", "_source_file", "_pipeline_run_id"]

TABLE_REGISTRY = [
    {
        "file"            : "bronze_purchase_orders.csv",
        "critical_id_cols": ["POID"],
        "expected_cols"   : [
            "POID","SupplierID","ProductID","OrderedQty","ReceivedQty",
            "UnitCost_USD","TotalCost_USD","Currency","ExchangeRate",
            "OrderDate","ExpectedDeliveryDate","ActualDeliveryDate",
            "Status","IncoTerms","PortOfLoading","FreightCost_USD",
            "BuyerName","ApprovalStatus","Notes",
        ],
        "row_regression_pct": 0.20,
    },
    {
        "file"            : "bronze_sales_orders.csv",
        "critical_id_cols": ["OrderID", "CustomerID"],
        "expected_cols"   : [
            "OrderID","CustomerID","ProductID","Division","OrderedQty",
            "UnitPrice_USD","LineTotal_USD","DiscountPct","OrderDate",
            "RequestedDeliveryDate","Status","SalesChannel","SalesRepID",
            "POReference","SpecialInstructions","IsRush","SourceSystem",
        ],
        "row_regression_pct": 0.10,
    },
    {
        "file"            : "bronze_products.csv",
        "critical_id_cols": ["ProductID"],
        "expected_cols"   : [
            "ProductID","ProductName","Category","SubCategory","UOM",
            "BasePrice_USD","StandardCost_USD","PrimarySupplierID",
            "HSCode","WeightPerUnit_KG","IsHazardous","Division",
            "ActiveFlag","IntroducedDate",
        ],
        "row_regression_pct": 0.05,
    },
    {
        "file"            : "bronze_customers.csv",
        "critical_id_cols": ["CustomerID"],
        "expected_cols"   : [
            "CustomerID","CustomerName","Segment","Division","Country",
            "State","City","Region","CreditLimit_USD","PaymentTerms",
            "SalesRepID","AccountManagerID","CustomerSince",
            "AnnualRevenueTier","ActiveFlag","PreferredCurrency","TaxExempt",
        ],
        "row_regression_pct": 0.05,
    },
    {
        "file"            : "bronze_inventory.csv",
        "critical_id_cols": ["InventoryID"],
        "expected_cols"   : [
            "InventoryID","ProductID","WarehouseID","StockQty","AllocatedQty",
            "AvailableQty","ReorderLevel","ReorderQty","UnitCost_USD",
            "TotalValue_USD","LastReceivedDate","LastIssuedDate","LotNumber",
            "ExpiryDate","BelowReorder","StorageLocation","LastUpdated",
        ],
        "row_regression_pct": 0.15,
    },
]

print(f"Tables    : {len(TABLE_REGISTRY)}")
print(f"Run ID    : {pipeline_run_id}")
print(f"Run at    : {run_at.isoformat()}")

Tables    : 5
Run ID    : 20260625_181725
Run at    : 2026-06-25T18:17:25.080607+00:00


## 2. Imports

`great-expectations` must be installed in the attached Fabric Environment.

In [10]:
!pip install great-expectations 

import great_expectations as gx
import great_expectations.expectations as gxe

print(f"great_expectations : {gx.__version__}")
print(f"pandas             : {pd.__version__}")

context = gx.get_context(mode="ephemeral")
print("GX context ready.")

great_expectations : 1.18.1
pandas             : 2.3.3
GX context ready.


## 3. Helpers

In [18]:
def load_bronze(filename: str) -> pd.DataFrame:
    path = BRONZE_PATH + filename
    try:
        df = pd.read_csv(path, dtype=str)
        print(f"  Loaded: {len(df):,} rows × {len(df.columns)} cols")
        return df
    except FileNotFoundError:
        raise FileNotFoundError(
            f"Bronze file missing: {path}\n"
            "Bronze ingestion notebook must succeed before this notebook runs."
        )


def load_baseline(filename: str) -> dict | None:
    path = BASELINE_PATH + filename.replace(".csv", "_baseline.json")
    try:
        return json.load(open(path))
    except FileNotFoundError:
        return None


def save_baseline(filename: str, row_count: int, col_names: list):
    """Only called after a fully clean run — never poison the baseline."""
    path = BASELINE_PATH + filename.replace(".csv", "_baseline.json")
    json.dump(
        {
            "row_count": row_count,
            "col_names": col_names,
            "saved_at" : run_at.isoformat(),
            "run_id"   : pipeline_run_id,
        },
        open(path, "w"), indent=2
    )
    print(f"  Baseline saved: {row_count:,} rows, {len(col_names)} cols")


def build_suite(suite_name: str, df: pd.DataFrame, cfg: dict, baseline: dict | None):
    """
    Builds the GX expectation suite for one table.
    Returns (suite, validation_definition).

    Idempotent: deletes any prior GX objects with the same name first so the
    cell can be re-run in the same notebook session without conflicts.
    """
    # Clean up prior objects from earlier runs in this kernel session.
    # validation_definition first (it references suite + data_source).
    for collection, name in [
        (context.validation_definitions, f"vd_{suite_name}"),
        (context.suites,                 suite_name),
        (context.data_sources,           f"ds_{suite_name}"),
    ]:
        try:
            collection.delete(name=name)
        except Exception:
            pass   # not present — nothing to clean up
    
    # Create a suite
    suite = context.suites.add(gx.ExpectationSuite(name=suite_name))
    # Defining where the data lives and how to read it
    ds    = context.data_sources.add_pandas(name=f"ds_{suite_name}")
    # Defining the data asset like which table
    da    = ds.add_dataframe_asset(name=f"asset_{suite_name}")
    # Defining how to slice the asset into batch for validation
    bd    = da.add_batch_definition_whole_dataframe(name=f"batch_{suite_name}")
    # It links a specific batch of data (bd) to a specific suite of expectations (suite)
    vd    = context.validation_definitions.add(
                gx.ValidationDefinition(name=f"vd_{suite_name}", data=bd, suite=suite)
            )

    # ── 1. COMPLETENESS ──────────────────────────────────────────────────────

    # Row count > 0  (GX 1.x has no "greater than" — use Between with high ceiling)
    suite.add_expectation(gxe.ExpectTableRowCountToBeBetween(min_value=1))

    # Row count regression vs baseline
    if baseline is not None:
        prev      = baseline["row_count"]
        threshold = cfg["row_regression_pct"]
        min_rows  = int(prev * (1 - threshold))
        # GX 1.x has no "greater than or equal" — use Between with min only
        suite.add_expectation(
            gxe.ExpectTableRowCountToBeBetween(min_value=min_rows)
        )
        print(f"  [regression] prev={prev:,}  min_allowed={min_rows:,}  threshold={threshold:.0%}")
    else:
        print(f"  [regression] no baseline yet — first run, skipping")

    # ── 2. SCHEMA ────────────────────────────────────────────────────────────

    # Expected column count = business cols + audit cols
    expected_total = len(cfg["expected_cols"]) + len(AUDIT_COLS)
    suite.add_expectation(gxe.ExpectTableColumnCountToEqual(value=expected_total))

    # Every expected business column must exist
    for col in cfg["expected_cols"]:
        suite.add_expectation(gxe.ExpectColumnToExist(column=col))

    # Every audit column must exist
    for col in AUDIT_COLS:
        suite.add_expectation(gxe.ExpectColumnToExist(column=col))

    # ── 3. HYGIENE ───────────────────────────────────────────────────────────

    # Critical ID columns — zero nulls
    for col in cfg["critical_id_cols"]:
        suite.add_expectation(gxe.ExpectColumnValuesToNotBeNull(column=col))

    # Audit columns — must be fully populated on every row
    for col in AUDIT_COLS:
        suite.add_expectation(gxe.ExpectColumnValuesToNotBeNull(column=col))

    return suite, vd


print("Helpers ready.")

Helpers ready.


## 4. Run validations

In [19]:
results = []

for cfg in TABLE_REGISTRY:
    fname      = cfg["file"]
    suite_name = fname.replace(".csv", "")

    print(f"\n{'='*60}")
    print(f"Validating: {fname}")
    print(f"{'='*60}")

    issues   = []   # FAIL-level — will halt pipeline
    warnings = []   # WARN-level — logged only

    try:
        # Load
        df       = load_bronze(fname)
        baseline = load_baseline(fname)

        # Schema drift vs baseline — logged as WARN, not a GX expectation
        # (extra columns don't fail the suite but Silver config may need updating)
        if baseline is not None:
            prev_cols = set(baseline.get("col_names", []))
            curr_cols = set(df.columns) - set(AUDIT_COLS)
            extra     = sorted(curr_cols - prev_cols)
            dropped   = sorted(prev_cols - curr_cols)
            if extra:
                msg = f"WARN [schema_extra] New columns since last run: {extra}"
                warnings.append(msg)
                print(f"  {msg}")
            if dropped:
                # Dropped columns will also be caught by GX ExpectColumnToExist
                # but flag it here too for a clear log message
                msg = f"WARN [schema_drop] Columns missing vs last run: {dropped}"
                warnings.append(msg)
                print(f"  {msg}")

        # Build and run GX suite
        suite, vd = build_suite(suite_name, df, cfg, baseline)
        vr        = vd.run(batch_parameters={"dataframe": df})

        total_exp  = len(vr.results)
        passed_exp = sum(1 for r in vr.results if r.success)
        failed_exp = total_exp - passed_exp

        print(f"  GX: {passed_exp}/{total_exp} expectations passed")

        if not vr.success:
            for r in vr.results:
                if not r.success:
                    exp_type = r.expectation_config.type
                    col      = r.expectation_config.kwargs.get("column", "(table-level)")
                    observed = r.result.get("observed_value", "n/a")
                    msg = f"FAIL [{exp_type}] col={col} observed={observed}"
                    issues.append(msg)
                    print(f"    ❌ {msg}")

        # Save baseline only on fully clean runs
        if not issues:
            data_cols = [c for c in df.columns if c not in AUDIT_COLS]
            save_baseline(fname, len(df), data_cols)

        results.append({
            "file"           : fname,
            "rows"           : len(df),
            "gx_total"       : total_exp,
            "gx_passed"      : passed_exp,
            "gx_failed"      : failed_exp,
            "suite_passed"   : vr.success and not issues,
            "issues"         : issues,
            "warnings"       : warnings,
            "run_at"         : run_at.isoformat(),
            "pipeline_run_id": pipeline_run_id,
        })

    except (FileNotFoundError, IOError) as e:
        print(f"  ERROR: {e}")
        results.append({
            "file": fname, "rows": 0,
            "gx_total": 0, "gx_passed": 0, "gx_failed": 0,
            "suite_passed": False,
            "issues": [f"FAIL [load_error] {e}"], "warnings": [],
            "run_at": run_at.isoformat(), "pipeline_run_id": pipeline_run_id,
        })
    except Exception as e:
        print(f"  UNEXPECTED ERROR: {e}")
        results.append({
            "file": fname, "rows": 0,
            "gx_total": 0, "gx_passed": 0, "gx_failed": 0,
            "suite_passed": False,
            "issues": [f"FAIL [exception] {e}"], "warnings": [],
            "run_at": run_at.isoformat(), "pipeline_run_id": pipeline_run_id,
        })

print(f"\n{'='*60}")
print("VALIDATION COMPLETE")
print(f"{'='*60}")


Validating: bronze_purchase_orders.csv
  Loaded: 1,248 rows × 22 cols
  [regression] no baseline yet — first run, skipping


Calculating Metrics:   0%|          | 0/25 [00:00<?, ?it/s]

  GX: 28/28 expectations passed
  Baseline saved: 1,248 rows, 19 cols

Validating: bronze_sales_orders.csv
  Loaded: 5,150 rows × 20 cols
  [regression] no baseline yet — first run, skipping


Calculating Metrics:   0%|          | 0/30 [00:00<?, ?it/s]

  GX: 27/27 expectations passed
  Baseline saved: 5,150 rows, 17 cols

Validating: bronze_products.csv
  Loaded: 42 rows × 17 cols
  [regression] no baseline yet — first run, skipping


...

  GX: 23/23 expectations passed
  Baseline saved: 42 rows, 14 cols

Validating: bronze_customers.csv
  Loaded: 72 rows × 20 cols
  [regression] no baseline yet — first run, skipping


Calculating Metrics:   0%|          | 0/25 [00:00<?, ?it/s]

  GX: 26/26 expectations passed
  Baseline saved: 72 rows, 17 cols

Validating: bronze_inventory.csv
  Loaded: 215 rows × 20 cols
  [regression] no baseline yet — first run, skipping


Calculating Metrics:   0%|          | 0/25 [00:00<?, ?it/s]

  GX: 26/26 expectations passed
  Baseline saved: 215 rows, 17 cols

VALIDATION COMPLETE


## 5. Write cumulative DQ log

In [20]:
log_path = DQ_LOG_PATH + "bronze_gx_log.csv"

rows = []
for r in results:
    events = [(m, "FAIL") for m in r["issues"]] +              [(m, "WARN") for m in r["warnings"]]
    if not events:
        events = [("All checks passed", "OK")]
    for msg, sev in events:
        rows.append({
            "pipeline_run_id": r["pipeline_run_id"],
            "run_at"         : r["run_at"],
            "file"           : r["file"],
            "rows_ingested"  : r["rows"],
            "gx_passed"      : r["gx_passed"],
            "gx_failed"      : r["gx_failed"],
            "severity"        : sev,
            "message"         : msg,
            "suite_passed"   : r["suite_passed"],
        })

new_df = pd.DataFrame(rows)
try:
    existing = pd.read_csv(log_path)
    combined = pd.concat([existing, new_df], ignore_index=True)
except FileNotFoundError:
    combined = new_df

combined.to_csv(log_path, index=False)
print(f"DQ log updated: {log_path}  ({len(combined):,} total rows)")
print()
print(new_df[["file","rows_ingested","gx_passed","gx_failed","suite_passed","severity"]].to_string(index=False))

DQ log updated: abfss://CNG_project@onelake.dfs.fabric.microsoft.com/bronze_lakehouse.Lakehouse/Files/dq_logs/bronze_gx_log.csv  (10 total rows)

                      file  rows_ingested  gx_passed  gx_failed  suite_passed severity
bronze_purchase_orders.csv           1248         28          0          True       OK
   bronze_sales_orders.csv           5150         27          0          True       OK
       bronze_products.csv             42         23          0          True       OK
      bronze_customers.csv             72         26          0          True       OK
      bronze_inventory.csv            215         26          0          True       OK


## 6. Pipeline halt decision

In [21]:
failed = [r for r in results if not r["suite_passed"]]
passed = [r for r in results if r["suite_passed"]]

print(f"Passed : {len(passed)}")
print(f"Failed : {len(failed)}")

if failed:
    detail = " | ".join(
        f"{r['file']}: {r['issues'][0] if r['issues'] else 'suite failed'}"
        for r in failed
    )
    raise Exception(
        f"BRONZE GX HALT — {len(failed)} table(s) failed. "
        f"Silver will NOT run. Details: {detail}"
    )

print("\n✅ All Bronze DQ checks passed. Safe to run Silver.")

Passed : 5
Failed : 0

✅ All Bronze DQ checks passed. Safe to run Silver.
